# 04 — Ablation analysis: does rank matter?

**Goal:** turn the run log into the answer. Run the missing ranks first (if not already done), then plot accuracy vs rank, accuracy vs trainable parameters, and training time — and write down what the plots actually show, including where the differences are just noise.

SST-2 validation is only 872 examples: a gap under ~1–2 points between two ranks is not a real win. This notebook says that explicitly wherever it applies.

In [ ]:
# Setup: run this first in Colab (or skip locally if requirements are installed)
# In Colab, clone the repo so src/ and experiments/ are available:
#   !git clone https://github.com/sujalgiriiitp-source/Lora---reproduction-.git
#   %cd Lora---reproduction-
# Then install dependencies (takes a few minutes the first time):
#   !pip install -q -r requirements.txt
import sys
print(sys.version)
try:
    import torch, transformers, peft, datasets
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
    print("transformers", transformers.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__)
except ImportError as e:
    print("Missing package:", e)
    print("Run: pip install -r requirements.txt")


## 1. Run the remaining ranks (skip if already in `runs.csv`)

r=4 and r=16, identical settings to r=8. Only the rank changes.

In [ ]:
# Run only the ranks missing from experiments/runs.csv:
!python src/train.py --mode lora --r 4  --epochs 3 --batch-size 32 --output runs/lora-r4  --run-id lora-r4
!python src/train.py --mode lora --r 16 --epochs 3 --batch-size 32 --output runs/lora-r16 --run-id lora-r16

## 2. Load the run log

Everything below reads `experiments/runs.csv`. If a row is missing, the plots will show it — do not fill in numbers by hand.

In [ ]:
import pandas as pd
runs = pd.read_csv("experiments/runs.csv")
# Keep only real full runs (exclude smoke tests) for the comparison:
real = runs[~runs["notes"].fillna("").str.contains("smoke", case=False)]
print(real[["run_id", "mode", "lora_r", "trainable_params", "accuracy", "f1_macro", "train_time_sec"]].to_string(index=False))

## 3. Plot 1 — Accuracy vs rank

Expectation from the paper: accuracy rises with rank, then flattens. The interesting finding is *where* it flattens — that is the rank you would actually choose.

In [ ]:
import matplotlib.pyplot as plt
lora = real[real["mode"] == "lora"].sort_values("lora_r")
full_acc = real[real["mode"] == "full"]["accuracy"].iloc[0] if (real["mode"] == "full").any() else None
plt.figure(figsize=(7, 4))
plt.plot(lora["lora_r"], lora["accuracy"], marker="o")
if full_acc is not None:
    plt.axhline(full_acc, linestyle="--", label=f"Full fine-tuning ({full_acc:.3f})")
plt.xlabel("LoRA rank r")
plt.ylabel("Validation accuracy")
plt.title("SST-2 accuracy vs LoRA rank (DistilBERT)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Plot 2 — Accuracy vs trainable parameters (the trade-off)

This is the plot that matters to a reader: how much accuracy does each parameter buy? A rank that costs 4x the parameters for +0.2 points is a bad trade, even if it is technically the best.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(real["trainable_params"], real["accuracy"], marker="o")
for _, row in real.iterrows():
    plt.annotate(row["run_id"], (row["trainable_params"], row["accuracy"]), textcoords="offset points", xytext=(5, 5))
plt.xscale("log")
plt.xlabel("Trainable parameters (log scale)")
plt.ylabel("Validation accuracy")
plt.title("Accuracy vs trainable parameters")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Write the conclusion (fill in after the real runs)

Answer in one paragraph each:

1. **Rank effect:** did accuracy rise from r=4 to r=16, and where did it flatten?
2. **vs full fine-tuning:** which rank, if any, came within ~1 point of the baseline — and at what fraction of the parameters?
3. **Cost:** how did training time change across ranks? (It should barely move — the adapters are tiny; if it moved a lot, something else changed.)
4. **What I would do with one more month:** second seed for the best rank, a second dataset (AG News), or LoRA on all linear layers?

Copy this conclusion into `report/report.md` Section 3 — with the numbers from the table above, not from memory.